# Perceptron clássico e classificador logístico em PyTorch

Este notebook separa explicitamente **dois classificadores lineares relacionados, mas diferentes**:

1. o **Perceptron clássico de Rosenblatt**, com ativação degrau e atualização baseada em erros de classificação;
2. o **classificador logístico** (uma unidade linear com sigmóide), treinado por minimização de entropia cruzada com gradientes.

Ambos usam a mesma transformação afim

$$z = \mathbf{w}^T\mathbf{x} + b,$$

mas **não usam a mesma regra de treinamento**.

> `nn.Linear + Sigmoid/BCE` implementa regressão logística binária, não o algoritmo clássico do Perceptron.

## Objetivos

| Conceito | Objetivo |
|---|---|
| Transf. afim | Entender $z=\mathbf{w}^T\mathbf{x}+b$ e a fronteira $z=0$ |
| Perceptron clássico | Entender a regra de atualização baseada em erros |
| Sigmóide + BCE | Entender por que aparece gradiente no classificador logístico |
| Convexidade | Distinguir o problema linear de redes multicamadas não convexas |
| Separabilidade linear | Entender por que AND é resolvível e XOR não é por uma única unidade linear |


## 1. Transformação afim e fronteira de decisão

A analogia com sinapses pode ser útil como recurso mnemônico, mas o modelo matemático abaixo é um **classificador linear**, não um modelo fisiologicamente plausível de um neurônio.

Para uma entrada $\mathbf{x}\in\mathbb{R}^d$, pesos $\mathbf{w}\in\mathbb{R}^d$ e intercepto $b\in\mathbb{R}$, definimos

$$z = \mathbf{w}^T\mathbf{x}+b.$$

O valor $z$ é o resultado de uma **transformação afim**. A equação

$$\mathbf{w}^T\mathbf{x}+b=0$$

define um hiperplano: a fronteira que separa os semiespaços $z<0$ e $z>0$.

O intercepto $b$ desloca essa fronteira. Um limiar diferente de zero pode sempre ser absorvido no intercepto.


In [ ]:
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn

# Configurações de visualização
plt.rcParams['figure.facecolor'] = '#0d1117'
plt.rcParams['axes.facecolor'] = '#161b22'
plt.rcParams['axes.edgecolor'] = '#30363d'
plt.rcParams['text.color'] = '#e6edf3'
plt.rcParams['axes.labelcolor'] = '#e6edf3'
plt.rcParams['xtick.color'] = '#8b949e'
plt.rcParams['ytick.color'] = '#8b949e'
plt.rcParams['grid.color'] = '#21262d'

torch.manual_seed(42)

# AND lógico: conjunto linearmente separável
X = torch.tensor([[0., 0.],
                  [0., 1.],
                  [1., 0.],
                  [1., 1.]])

y01 = torch.tensor([[0.], [0.], [0.], [1.]])
y_pm = 2 * y01.flatten() - 1  # rótulos {-1, +1} para o Perceptron clássico

print('Dados do AND:')
for xi, yi in zip(X, y01):
    print(f'x={xi.tolist()}  y={int(yi.item())}')


## 2. Da transformação afim à decisão: degrau versus sigmóide

O **Perceptron clássico** usa uma decisão discreta:

$$\hat y_{\text{perc}} = \begin{cases} 1 & \text{if } z \ge 0 \\ 0 & \text{if } z < 0 \end{cases}$$

Já a regressão logística usa a função sigmóide

$$\sigma(z)=\frac{1}{1+e^{-z}},$$

interpretando $\sigma(z)$ como $P(y=1\mid\mathbf{x})$ sob o modelo logístico.

Se classificarmos a saída logística com limiar $0.5$, então

$$\sigma(z)\ge 0.5 \iff z\ge 0.$$

Portanto, **os dois modelos podem induzir a mesma fronteira de decisão linear**, embora suas saídas e regras de treinamento sejam diferentes.

A função degrau não fornece um gradiente útil para treinamento por retropropagação: sua derivada é zero quase em toda parte e não é definida no limiar. A sigmóide é diferenciável.


In [ ]:
z_range = torch.linspace(-6, 6, 300)
step = (z_range >= 0).float()
sigmoid = torch.sigmoid(z_range)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].plot(z_range.numpy(), step.numpy(), lw=2.5)
axes[0].axvline(0, linestyle='--', alpha=0.7)
axes[0].set_title('Degrau: Perceptron clássico')
axes[0].set_xlabel('z')
axes[0].set_ylabel('saída')
axes[0].grid(True, alpha=0.3)

axes[1].plot(z_range.numpy(), sigmoid.numpy(), lw=2.5)
axes[1].axvline(0, linestyle='--', alpha=0.7)
axes[1].axhline(0.5, linestyle='--', alpha=0.7)
axes[1].set_title('Sigmoid: regressão logística')
axes[1].set_xlabel('z')
axes[1].set_ylabel('σ(z)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 3. Algoritmo clássico do Perceptron

Usaremos agora rótulos $y_i\in\{-1,+1\}$. Para uma amostra $(\mathbf{x}_i,y_i)$, o Perceptron verifica o sinal da margem

$$y_i(\mathbf{w}^T\mathbf{x}_i+b).$$

Se

$$y_i(\mathbf{w}^T\mathbf{x}_i+b)\le 0,$$

a amostra está errada ou exatamente sobre a fronteira. A atualização clássica é

$$\mathbf{w}\leftarrow\mathbf{w}+\eta y_i\mathbf{x}_i,$$

$$b\leftarrow b+\eta y_i,$$

onde $\eta>0$ é a taxa de aprendizado.

Essa regra é **mistake-driven**: parâmetros só mudam quando há erro (ou margem zero). Ela não precisa de `loss.backward()` nem de uma ativação diferenciável.

Uma interpretação útil é que a regra corresponde a um passo de subgradiente sobre o critério do Perceptron

$$\ell_i(\mathbf{w},b)=\max\left(0,-y_i(\mathbf{w}^T\mathbf{x}_i+b)\right),$$

mas isso não é o mesmo que treinar uma sigmoid com BCE.

### Teorema de convergência do Perceptron

Se os dados forem linearmente separáveis, o algoritmo clássico com taxa positiva fixa comete um número finito de erros. Se os dados não forem linearmente separáveis, essa garantia desaparece e as atualizações podem continuar indefinidamente.


In [ ]:
def treinar_perceptron_classico(X_, y_, eta=1.0, max_epocas=50):
    w = torch.zeros(X_.shape[1])
    b = torch.tensor(0.0)
    erros_por_epoca = []

    for _ in range(max_epocas):
        erros = 0
        for xi, yi in zip(X_, y_):
            z = torch.dot(w, xi) + b
            if yi * z <= 0:
                w = w + eta * yi * xi
                b = b + eta * yi
                erros += 1

        erros_por_epoca.append(erros)
        if erros == 0:
            break

    return w, b, erros_por_epoca

w_perc, b_perc, erros_perc = treinar_perceptron_classico(X, y_pm)

print('Perceptron clássico treinado sem autograd:')
print(f'w = {w_perc.tolist()}')
print(f'b = {b_perc.item():.3f}')
print(f'erros por época = {erros_perc}')


In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 5.5))
seen_classes = set()

for xi, yi in zip(X.numpy(), y01.numpy().flatten()):
    class_label = int(yi)
    if class_label not in seen_classes:
        ax.scatter(xi[0], xi[1], s=180, edgecolors='white', linewidths=1.5,
                   label=f'classe {class_label}')
        seen_classes.add(class_label)
    else:
        ax.scatter(xi[0], xi[1], s=180, edgecolors='white', linewidths=1.5)

x_line = np.linspace(-0.5, 1.5, 200)
if abs(w_perc[1].item()) > 1e-12:
    y_line = -(w_perc[0].item() * x_line + b_perc.item()) / w_perc[1].item()
    ax.plot(x_line, y_line, '--', lw=2, label=r'$w^Tx+b=0$')

ax.set_xlim(-0.5, 1.5)
ax.set_ylim(-0.5, 1.5)
ax.set_xlabel('x₁')
ax.set_ylabel('x₂')
ax.set_title('Fronteira aprendida pelo Perceptron clássico')
ax.grid(True, alpha=0.25)
ax.legend()
plt.tight_layout()
plt.show()


## 4. Alternativa suave: regressão logística

Agora trocamos deliberadamente de algoritmo.

Definimos

$$p_i=\sigma(z_i),\qquad z_i=\mathbf{w}^T\mathbf{x}_i+b,$$

com a entropia cruzada binária (BCE)

$$L=-\frac1n\sum_i \left[y_i\log p_i+(1-y_i)\log(1-p_i)\right].$$

Para uma única amostra, a derivada em relação ao logit $z$ é especialmente simples:

$$\frac{\partial \ell}{\partial z}=\sigma(z)-y.$$

Logo,

$$\nabla_{\mathbf w}\ell=(\sigma(z)-y)\mathbf{x},\qquad
\frac{\partial \ell}{\partial b}=\sigma(z)-y.$$

É **aqui** que o gradiente entra: adotamos uma função de perda suave e diferenciável para um modelo logístico.

Em PyTorch, `nn.BCEWithLogitsLoss()` é preferível a aplicar `Sigmoid` e depois `BCELoss`, porque combina as duas operações de forma numericamente mais estável. O modelo abaixo retorna **logits**, não probabilidades.


In [ ]:
class ClassificadorLogistico(nn.Module):
    def __init__(self, n_entradas):
        super().__init__()
        self.linear = nn.Linear(n_entradas, 1)

    def forward(self, x_):
        return self.linear(x_)  # logits z = w^T x + b


torch.manual_seed(0)
modelo = ClassificadorLogistico(n_entradas=2)
criterio = nn.BCEWithLogitsLoss()
lambda_l2 = 0.05

# O otimizador chama-se SGD, mas como usamos o conjunto inteiro em cada passo,
# o procedimento abaixo é gradiente em batch completo, não SGD estocástico.
otimizador = torch.optim.SGD(modelo.parameters(), lr=0.5)

historico_objetivo = []
historico_bce = []
n_epocas = 200

for _ in range(n_epocas):
    logits = modelo(X)
    bce = criterio(logits, y01)
    reg = 0.5 * lambda_l2 * modelo.linear.weight.pow(2).sum()
    objetivo = bce + reg

    otimizador.zero_grad()
    objetivo.backward()
    otimizador.step()

    historico_bce.append(bce.item())
    historico_objetivo.append(objetivo.item())

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(historico_bce, label='BCE')
ax.plot(historico_objetivo, label='BCE + regularização L2')
ax.set_xlabel('Época')
ax.set_ylabel('Valor')
ax.set_title('Treinamento do classificador logístico')
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()


## 5. Geometria do problema: convexidade, separabilidade e mínimos

Para um classificador logístico **linear**, a BCE é uma função convexa de $(\mathbf{w},b)$. Portanto, este problema não possui os “maus mínimos locais” típicos de modelos multicamadas não convexos.

A regularização L2 adiciona

$$\frac{\lambda}{2}\|\mathbf{w}\|_2^2$$

ao objetivo. Além de controlar a magnitude dos pesos, ela fornece um ótimo finito para o caso usado neste lab.

Isso importa porque o conjunto AND é **perfeitamente separável**. Na regressão logística sem regularização, a BCE pode tender a zero enquanto $\|\mathbf{w}\|$ cresce sem limite; o ínfimo é zero, mas em geral não há um minimizador finito. Isso não é um mínimo local ruim — é uma propriedade diferente do problema separável.

Consequências para este notebook:

- Momentum e Adam podem mudar velocidade e trajetória de convergência;
- eles não são necessários para “escapar de mínimos locais” neste modelo linear convexo;
- mínimos locais, pontos de sela e paisagens fortemente não convexas tornam-se relevantes quando introduzimos, por exemplo, **camadas ocultas**.


In [ ]:
# Uma fatia do objetivo real do classificador logístico, mantendo b fixo.
# Como o objetivo completo é convexo, qualquer fatia afim também é convexa.

w1 = np.linspace(-5, 5, 120)
w2 = np.linspace(-5, 5, 120)
W1, W2 = np.meshgrid(w1, w2)
b_vis = -2.0

X_np = X.numpy()
y_np = y01.numpy().flatten()
J = np.zeros_like(W1)

for xi, yi in zip(X_np, y_np):
    logits = W1 * xi[0] + W2 * xi[1] + b_vis
    J += np.logaddexp(0.0, logits) - yi * logits

J /= len(X_np)
J += 0.5 * lambda_l2 * (W1 ** 2 + W2 ** 2)

fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection='3d')
ax.plot_surface(W1, W2, J, cmap='viridis', alpha=0.9, linewidth=0)
ax.set_xlabel('w₁')
ax.set_ylabel('w₂')
ax.set_zlabel('Objetivo')
ax.set_title('Fatia convexa de BCE + L2 (b fixo)')
plt.tight_layout()
plt.show()


## 6. Comparando otimizadores sem atribuir o efeito a mínimos locais

A comparação abaixo usa exatamente o mesmo objetivo convexo e a mesma inicialização.

Há também uma distinção terminológica importante: `torch.optim.SGD` implementa a regra de atualização do SGD, mas o treinamento só é **estocástico** se o gradiente for estimado com amostras ou minibatches. Como usamos os quatro pontos do AND em cada iteração, cada passo usa o gradiente do batch completo.

Assim, neste experimento comparamos diferentes **regras de atualização**, não diferentes mecanismos para escapar de mínimos locais.


In [ ]:
def objetivo_logistico(m):
    logits = m(X)
    bce = criterio(logits, y01)
    reg = 0.5 * lambda_l2 * m.linear.weight.pow(2).sum()
    return bce + reg


otimizadores_config = {
    'GD via SGD (lr=0.05)': lambda p: torch.optim.SGD(p, lr=0.05),
    'GD via SGD (lr=0.5)': lambda p: torch.optim.SGD(p, lr=0.5),
    'Momentum': lambda p: torch.optim.SGD(p, lr=0.1, momentum=0.9),
    'Adam': lambda p: torch.optim.Adam(p, lr=0.05),
}

resultados = {}

for nome, construir_otimizador in otimizadores_config.items():
    torch.manual_seed(0)
    m = ClassificadorLogistico(n_entradas=2)
    opt = construir_otimizador(m.parameters())
    hist = []

    for _ in range(300):
        obj = objetivo_logistico(m)
        opt.zero_grad()
        obj.backward()
        opt.step()
        hist.append(obj.item())

    resultados[nome] = hist

fig, ax = plt.subplots(figsize=(9, 5))
for nome, hist in resultados.items():
    ax.plot(hist, lw=2, label=f'{nome}: {hist[-1]:.4f}')

ax.set_xlabel('Iteração')
ax.set_ylabel('BCE + L2')
ax.set_title('Regras de atualização no mesmo problema convexo')
ax.set_yscale('log')
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()


## 7. Avaliação do classificador logístico

O modelo retorna logits. Para obter valores no intervalo $(0,1)$, aplicamos sigmoid apenas na avaliação:

$$p=\sigma(z).$$

Com limiar $0.5$, a decisão é equivalente a testar $z\ge 0$.

Neste conjunto de quatro pontos determinísticos, esses valores são úteis para visualizar a margem do modelo, mas não devem ser interpretados como uma demonstração de calibração probabilística.


In [ ]:
torch.manual_seed(0)
modelo_final = ClassificadorLogistico(n_entradas=2)
otimizador_final = torch.optim.Adam(modelo_final.parameters(), lr=0.05)

for _ in range(500):
    obj = objetivo_logistico(modelo_final)
    otimizador_final.zero_grad()
    obj.backward()
    otimizador_final.step()

with torch.no_grad():
    logits_finais = modelo_final(X)
    probabilidades = torch.sigmoid(logits_finais)
    predicoes = (logits_finais >= 0).float()

print('Predições finais:')
print(f"{'x1':>4} {'x2':>4} {'y':>4} {'logit':>10} {'sigmoid':>10} {'classe':>8}")
print('─' * 50)
for xi, yi, zi, pi, ci in zip(X, y01, logits_finais, probabilidades, predicoes):
    print(f'{xi[0].item():>4.0f} {xi[1].item():>4.0f} {yi.item():>4.0f} '
          f'{zi.item():>10.4f} {pi.item():>10.4f} {ci.item():>8.0f}')

fig, ax = plt.subplots(figsize=(7, 6))
xx, yy = np.meshgrid(np.linspace(-0.5, 1.5, 200),
                     np.linspace(-0.5, 1.5, 200))
grid = torch.tensor(np.c_[xx.ravel(), yy.ravel()], dtype=torch.float32)

with torch.no_grad():
    zz = torch.sigmoid(modelo_final(grid)).reshape(xx.shape).numpy()

contour = ax.contourf(xx, yy, zz, levels=np.linspace(0, 1, 51), cmap='RdYlGn', alpha=0.7)
ax.contour(xx, yy, zz, levels=[0.5], colors='white', linewidths=2, linestyles='--')
plt.colorbar(contour, ax=ax, label='σ(z)')

seen_classes = set()
for xi, yi in zip(X.numpy(), y01.numpy().flatten()):
    class_label = int(yi)
    if class_label not in seen_classes:
        ax.scatter(xi[0], xi[1], s=190, edgecolors='white', linewidths=1.5,
                   label=f'classe {class_label}')
        seen_classes.add(class_label)
    else:
        ax.scatter(xi[0], xi[1], s=190, edgecolors='white', linewidths=1.5)

ax.set_xlabel('x₁')
ax.set_ylabel('x₂')
ax.set_title('Regressão logística: fronteira σ(z)=0.5 ⇔ z=0')
ax.grid(True, alpha=0.2)
ax.legend()
plt.tight_layout()
plt.show()

print('\nParâmetros finais:')
print(f'w = {modelo_final.linear.weight.detach().flatten().tolist()}')
print(f'b = {modelo_final.linear.bias.detach().item():.4f}')


## 8. Limite de uma única unidade linear: XOR

Tanto o Perceptron clássico quanto a regressão logística acima usam uma única fronteira

$$\mathbf{w}^T\mathbf{x}+b=0.$$

Por isso, ambos só podem representar classificações **linearmente separáveis**.

A sigmoid é uma função não linear de $z$, mas, com uma única unidade e limiar $0.5$, a fronteira de classificação continua sendo um hiperplano. A não linearidade da sigmoid **não transforma uma única unidade logística em um classificador com fronteira não linear**.

O XOR é o exemplo clássico: seus rótulos não podem ser separados por uma única reta em 2D. Para representar XOR, precisamos mudar a classe de funções — por exemplo, usando uma MLP (*multilayer perceptron*) com camada(s) oculta(s) e ativações não lineares.


In [ ]:
X_xor = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y_xor = np.array([0, 1, 1, 0])

fig, ax = plt.subplots(figsize=(5.5, 5))
for xi, yi in zip(X_xor, y_xor):
    ax.scatter(xi[0], xi[1], s=190, edgecolors='white', linewidths=1.5)
    ax.text(xi[0] + 0.04, xi[1] + 0.04, f'y={yi}')

ax.set_xlim(-0.3, 1.3)
ax.set_ylim(-0.3, 1.3)
ax.set_xlabel('x₁')
ax.set_ylabel('x₂')
ax.set_title('XOR: não linearmente separável')
ax.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()


## 9. Resumo conceitual

| Aspecto | Perceptron clássico | Regressão logística |
|---|---|---|
| Escore | $z=\mathbf{w}^T\mathbf{x}+b$ | $z=\mathbf{w}^T\mathbf{x}+b$ |
| Saída | classe via degrau | $\sigma(z)$ |
| Rótulos convenientes | $\{-1,+1\}$ | $\{0,1\}$ |
| Treinamento canônico | atualização em erros | minimização de BCE |
| Gradiente suave | não é necessário | sim |
| `loss.backward()` | não | sim |
| Fronteira com limiar padrão | $z=0$ | $\sigma(z)=0.5\iff z=0$ |
| Geometria do objetivo | critério do Perceptron é convexo, não suave | BCE linear é convexa e suave |
| Dados separáveis | convergência em número finito de erros | BCE sem regularização pode não ter minimizador finito |
| XOR com uma unidade | não resolve | não resolve |

### Dois caminhos que não devem ser confundidos

```text
                    z = wᵀx + b
                    /         \\
                   /           \\
          degrau /               \\ sigmoid
                /                 \\
       classe 0/1                  probabilidade
           |                            |
 erro de classificação                BCE
           |                            |
 regra do Perceptron              gradiente/autograd
           |                            |
       atualização                    otimizador
```

O gradiente aparece no segundo caminho porque escolhemos uma **surrogate loss diferenciável**. Ele não é requisito do algoritmo clássico do Perceptron.


## Referências conceituais

- Rosenblatt, F. (1958). *The Perceptron: A Probabilistic Model for Information Storage and Organization in the Brain*.
- Novikoff, A. B. J. (1962). *On Convergence Proofs on Perceptrons*.
- Bishop, C. M. (2006). *Pattern Recognition and Machine Learning* — classificação linear e regressão logística.
- PyTorch: `nn.Linear`, `nn.BCEWithLogitsLoss`, `torch.optim.SGD` e `torch.optim.Adam`.

---
*Versão revisada para separar explicitamente Perceptron clássico, regressão logística e otimização convexa.*
